# VLM: изображение и текст

Патчи изображения, проектор и проверка зависимости от картинки.

> Все примеры этого урока работают на CPU, не скачивают модели и используют искусственные данные. Они объясняют механизм; качество настоящего обучения проверяется отдельно.

## Что важно понять

VLM связывает визуальное представление с текстовым. В простом варианте изображение разбивается на патчи, кодируется и проецируется в пространство признаков. Реальные готовые VLM требуют родной processor, формат диалога и правила image tokens.

Ниже маленькая fusion-модель на искусственной задаче: ответ зависит от яркости картинки и текстового признака. Это классификатор для понимания механизма, а не генеративная VLM. В VLM Studio можно обучать поддерживаемые готовые модели или собственную архитектуру из Research Lab; проверяйте доступность SFT/GRPO и backend до запуска.

In [ ]:
%matplotlib inline
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| device: CPU')

## Запустите эксперимент

Запускайте ячейки по порядку. Код можно менять; Shift + Enter выполняет текущую ячейку.

In [ ]:
images = torch.rand(64, 3, 8, 8)
instruction = torch.randint(0, 2, (64,))
# Shift brightness strongly enough that the task is visible in a tiny example.
images[:32] *= 0.2; images[32:] = 0.8 + images[32:] * 0.2
labels = ((images.mean((1, 2, 3)) > 0.5).long() ^ instruction)
class Fusion(nn.Module):
    def __init__(self):
        super().__init__()
        self.patch = nn.Conv2d(3, 8, kernel_size=4, stride=4)
        self.text = nn.Embedding(2, 4)
        self.head = nn.Sequential(nn.Linear(12, 24), nn.ReLU(), nn.Linear(24, 2))
    def forward(self, image, text):
        visual = self.patch(image).mean((2, 3))
        return self.head(torch.cat([visual, self.text(text)], -1))
model = Fusion(); optimizer = torch.optim.AdamW(model.parameters(), lr=0.03)
for step in range(80):
    loss = F.cross_entropy(model(images, instruction), labels)
    optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
with torch.no_grad():
    accuracy = (model(images, instruction).argmax(-1) == labels).float().mean()
    image_effect = (model(images, instruction) - model(torch.zeros_like(images), instruction)).abs().mean()
print('training accuracy:', accuracy.item(), 'image ablation delta:', image_effect.item())

## Попробуйте сами

Обнулите изображение до обучения. Сравните accuracy и image ablation delta. Затем сделайте отдельную validation выборку: обучение и оценка на одном наборе скрывают переобучение.

Сохраните копию ноутбука и запишите, что изменили, чего ожидали и что измерили.

## Продолжить в студии

Кнопка «Открыть раздел студии» над уроком ведёт к связанным инструментам. Учебный код сам не запускает большие тренировки. Перед реальным запуском проверьте данные, ресурсы и поддержку метода у выбранной модели.

## Источники

- [ViT](https://arxiv.org/abs/2010.11929)